In [ ]:
import os
import json
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from torchvision.transforms import RandAugment, AugMix
from PIL import Image
from tqdm.auto import tqdm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [ ]:
# Paths
DATA_DIR = "./data/tiny-imagenet-200"
CHECKPOINT_DIR = "./checkpoints"
RESULTS_DIR = "./results"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Hyperparams
NUM_CLASSES = 200
IMG_SIZE = 64
BATCH_SIZE = 256
NUM_EPOCHS = 100
LR = 0.1
MOMENTUM = 0.9
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 4

GPU_ID = 6

if torch.cuda.is_available():
    DEVICE = torch.device(f"cuda:{GPU_ID}")
    torch.cuda.set_device(DEVICE)
    print(f"Device: {DEVICE}  ({torch.cuda.get_device_name(DEVICE)})")
    print(f"GPUs available: {torch.cuda.device_count()}")

    # A100 only
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
else:
    DEVICE = torch.device("cpu")

In [ ]:
def reorganize_val_folder(val_dir):
    ann_file = os.path.join(val_dir, "val_annotations.txt")
    images_dir = os.path.join(val_dir, "images")
    if not os.path.exists(ann_file) or not os.path.isdir(images_dir):
        print("Organized")
        return
    print("Reorganizing")
    with open(ann_file) as f:
        for line in f:
            parts = line.strip().split("\t")
            fname, class_id = parts[0], parts[1]
            dst_dir = os.path.join(val_dir, class_id, "images")
            os.makedirs(dst_dir, exist_ok=True)
            src = os.path.join(images_dir, fname)
            dst = os.path.join(dst_dir, fname)
            if os.path.exists(src):
                os.rename(src, dst)

    # Clean up empty images/ dir
    if os.path.isdir(images_dir) and not os.listdir(images_dir):
        os.rmdir(images_dir)
    print("Cleaned")

reorganize_val_folder(os.path.join(DATA_DIR, "val"))


class TinyImageNet(Dataset):
    def __init__(self, root_dir, transform=None):
        self.transform = transform
        self.samples = []
        classes = sorted(
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d)) and d.startswith("n")
        )
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        for cls in classes:
            img_dir = os.path.join(root_dir, cls, "images")
            if not os.path.isdir(img_dir):
                img_dir = os.path.join(root_dir, cls)
            for fname in sorted(os.listdir(img_dir)):
                if fname.lower().endswith((".jpeg", ".jpg", ".png")):
                    self.samples.append(
                        (os.path.join(img_dir, fname), self.class_to_idx[cls])
                    )
        print(f"Loaded {len(self.samples):,} images · {len(classes)} classes · {root_dir}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR = os.path.join(DATA_DIR, "val")

In [ ]:
NORMALIZE = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225],
)

augmentation_configs = {
    "baseline": transforms.Compose([
        transforms.RandomCrop(64, padding=8),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        NORMALIZE,
    ]),
    "randaugment": transforms.Compose([
        transforms.RandomCrop(64, padding=8),
        transforms.RandomHorizontalFlip(),
        RandAugment(num_ops=2, magnitude=9),
        transforms.ToTensor(),
        NORMALIZE,
    ]),
    "augmix": transforms.Compose([
        transforms.RandomCrop(64, padding=8),
        transforms.RandomHorizontalFlip(),
        AugMix(),
        transforms.ToTensor(),
        NORMALIZE,
    ]),
}

val_transform = transforms.Compose([
    transforms.ToTensor(),
    NORMALIZE,
])

print("Strategies:", list(augmentation_configs.keys()))

In [ ]:
def create_model(num_classes=NUM_CLASSES):
    model = models.resnet18(weights=None)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(512, num_classes)
    return model

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(loader, desc="  train", leave=False)
    for images, labels in pbar:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct += logits.argmax(1).eq(labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix(loss=f"{loss.item():.3f}", acc=f"{100*correct/total:.1f}%")
    return total_loss / total, 100.0 * correct / total


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        logits = model(images)
        loss = criterion(logits, labels)
        total_loss += loss.item() * images.size(0)
        correct += logits.argmax(1).eq(labels).sum().item()
        total += labels.size(0)
    return total_loss / total, 100.0 * correct / total

In [ ]:
def train_model(strategy_name, train_transform):
    train_ds = TinyImageNet(TRAIN_DIR, transform=train_transform)
    val_ds = TinyImageNet(VAL_DIR, transform=val_transform)
    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
    )

    model = create_model().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": []}
    best_val_acc = 0.0

    for epoch in range(1, NUM_EPOCHS + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        vl_loss, vl_acc = validate(model, val_loader, criterion, DEVICE)
        scheduler.step()

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(vl_loss)
        history["val_acc"].append(vl_acc)
        history["lr"].append(optimizer.param_groups[0]["lr"])

        elapsed = time.time() - t0
        print(
            f"Epoch {epoch:3d}/{NUM_EPOCHS} │ "
            f"train {tr_loss:.4f} / {tr_acc:.1f}% │ "
            f"val {vl_loss:.4f} / {vl_acc:.1f}% │ "
            f"lr {optimizer.param_groups[0]['lr']:.5f} │ "
            f"{elapsed:.0f}s"
        )

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "val_acc": vl_acc,
                    "strategy": strategy_name,
                },
                os.path.join(CHECKPOINT_DIR, f"best_{strategy_name}.pth"),
            )
            print(f"  ↑ new best val acc: {vl_acc:.2f}%")

    torch.save(
        {
            "epoch": NUM_EPOCHS,
            "model_state_dict": model.state_dict(),
            "val_acc": vl_acc,
            "best_val_acc": best_val_acc,
            "strategy": strategy_name,
        },
        os.path.join(CHECKPOINT_DIR, f"final_{strategy_name}.pth"),
    )
    
    # Save history
    with open(os.path.join(RESULTS_DIR, f"history_{strategy_name}.json"), "w") as f:
        json.dump(history, f)

    print(f"\nFinished {strategy_name} — best val acc = {best_val_acc:.2f}%")
    return history, best_val_acc